<a href="https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice: Logistic Regression, then Random Forest.

My question is "will this content decline?" — a yes/no outcome with an observed label
(is_declining_label = trend_direction == 'down', built in 01_prepare_features.py). Per the
skill's table, that maps directly to "yes/no with an observed label → Logistic Regression,
then Random Forest." Logistic Regression gives a readable coefficient-based baseline model;
Random Forest is added second to see whether it earns its extra complexity over the linear
model, not assumed superior by default.

I did not use trend_direction or trend_pct as features — trend_direction IS the label, and
trend_pct is what trend_direction is derived from. Using either would mean predicting the
label from itself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: grouped by client_id, not random row-level split.

The data skill is explicit that content_id/client_id are pseudonyms for grouping/joining
only, never features — and the same logic extends to splitting. If two pages from the same
client land in both train and test, the model can learn client-specific quirks (a client's
house style, their typical CTR range) rather than generalizable decline signals, and the
test score becomes optimistic. GroupShuffleSplit on client_id keeps every client's content
entirely in one side of the split.

I did not add a time-aware split on top of this — the prepared feature file doesn't carry
an as-of date per row (all rows are one fixed 90-day trailing snapshot), so there's no time
axis to split on beyond the *_last_30d/_prev_30d columns already baked in as features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Train + compare: same test split, same precision@K metric as the Week-4 baseline, computed
in this notebook run so the numbers are directly comparable.

| k   | base_rate | baseline | logreg | rf   |
|-----|-----------|----------|--------|------|
| 20  | 0.517     | 0.20     | 0.80   | 0.45 |
| 50  | 0.517     | 0.28     | 0.86   | 0.60 |
| 100 | 0.517     | 0.32     | 0.71   | 0.64 |

Both models beat the baseline at every K. Logistic Regression outperforms Random Forest at
every K — a genuinely interesting result, since the more flexible model didn't earn its
extra complexity here. LogReg's precision peaks at K=50 (0.86) then dips at K=100 (0.71),
while RF trends the opposite direction, climbing from 0.45 to 0.64 as K grows. That crossover
is itself informative: LogReg is very confident about a small set of clear cases, while RF
is more consistent but less decisive at the top of the ranking.

In [2]:
import os, subprocess

if not os.path.exists("/content/flyrank-ml"):
    subprocess.run(["git", "clone", "https://github.com/Hashir9099/flyrank-ml.git"], check=True)

%cd /content/flyrank-ml

/content/flyrank-ml


In [6]:
import subprocess, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# --- 1. Build the prepared feature file (defines is_declining_label, drops leakage-prone rows) ---
result = subprocess.run(["python", "scripts/01_prepare_features.py"], capture_output=True, text=True)
print(result.stdout[-500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

feature_path = glob.glob("data/processed/refresh_feature_vector.csv") or glob.glob("**/refresh_feature_vector.csv", recursive=True)
assert feature_path, "Could not find refresh_feature_vector.csv — check script output above."
feat = pd.read_csv(feature_path[0])
print(f"Loaded {len(feat)} rows, {feat['is_declining_label'].mean():.3f} decline rate")

# --- 2. Feature list, explicitly excluding leakage columns ---
candidate_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "has_clicks", "has_ai_sessions", "measurable_opportunity",
]
LEAKAGE_COLS = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
numeric_features = [c for c in candidate_numeric if c in feat.columns and c not in LEAKAGE_COLS]
dropped = [c for c in candidate_numeric if c not in feat.columns]
print("Using numeric features:", numeric_features)
if dropped:
    print("Not found in this file (skipped):", dropped)

X = feat[numeric_features].copy()
y = feat["is_declining_label"].copy()
groups = feat["client_id"]

# --- 3. Grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train)} rows, {X_train.shape[0]} | Test: {len(X_test)} rows")
print(f"Client overlap check (should be 0): {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

# --- 4. Train models ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # trees don't need scaling
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- 5. Rebuild the Week-4 baseline score on the SAME test rows ---
test_df = feat.iloc[test_idx].copy()
stale_threshold = feat["days_since_last_update"].quantile(0.75)  # threshold fit on full data, same as w04
stale = test_df["days_since_last_update"] >= stale_threshold
visible = test_df["impressions_90d"] >= 500
test_df["position_bucket"] = pd.qcut(test_df["avg_position"].replace(0, np.nan), 10, duplicates="drop")
median_ctr = test_df.groupby("position_bucket", observed=True)["ctr"].transform("median")
low_ctr = test_df["ctr"] < median_ctr
slipping = test_df["avg_position"] > 20
baseline_score = (stale & visible).astype(int) * test_df["impressions_90d"] * (1 + low_ctr.astype(int) + slipping.astype(int))

# --- 6. Precision@K comparison table ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
rows = []
for k in [20, 50, 100]:
    rows.append({
        "k": k,
        "base_rate": round(base_rate, 3),
        "baseline_precision": round(precision_at_k(baseline_score.values, y_test.values, k), 3),
        "logreg_precision": round(precision_at_k(logreg_scores, y_test.values, k), 3),
        "rf_precision": round(precision_at_k(rf_scores, y_test.values, k), 3),
    })
comparison_table = pd.DataFrame(rows)
print(comparison_table.to_string(index=False))

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml/data/processed/refresh_feature_vector.csv

Loaded 30000 rows, 0.542 decline rate
Using numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']
Train: 22885 rows, 22885 | Test: 7115 rows
Client overlap check (should be 0): 0
  k  base_rate  baseline_precision  logreg_precision  rf_precision
 20      0.517                0.20              0.80          0.45
 50      0.517                0.28              0.86          0.60
100  

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Errors and interpretation:

Comparison table shows both models beat the Week-4 baseline (0.20-0.32) by a wide margin —
LogReg reaches 0.71-0.86 precision, RF reaches 0.45-0.64. LogReg wins at every K, an
interesting result: the simpler, linear model outperformed the more flexible one, so the
extra complexity of Random Forest wasn't earned here.

Top features (Random Forest permutation importance): days_with_impressions (0.019),
avg_position (0.010), log_clicks_90d (0.004), log_sessions_90d (0.004), ctr (0.003).
No single feature dominates — the sharp drop-off after impressions_last_30d/prev_30d in
the earlier (leaky) run is gone, and importance now decays gradually. This is the shape
I'd expect from a real, distributed signal rather than one column doing all the work.

days_with_impressions leading makes sense: a page that shows up in search results
consistently across the 90-day window is plausibly more stable than one with sparse,
intermittent visibility — consistency itself may be informative about whether a page is
"alive" in the index. avg_position ranking second also fits — position is directly tied
to how a page competes, and small differences at page-1 positions can matter a lot.

Classification report: accuracy 0.569, with the model recalling the declining class (0.733)
much better than the non-declining class (0.394) — it's biased toward predicting "decline."
Given the base rate is roughly balanced (0.517), this asymmetry is worth watching rather
than dismissing as noise.

3 most confidently wrong cases: all three share a pattern I hadn't accounted for —
avg_position of 0.0-5.0 with impressions_90d of only 1-2 and ctr of 0.0. Per the data
dictionary, avg_position = 0 specifically means "no data," not an actual top rank — so
content_7bc32bc1df59 (avg_position=0.0) is a near-empty row: almost no impressions, no
clicks, and essentially no position signal at all. The model predicted "not declining"
with high confidence (score ~0.05-0.09) on all three, but the true label says declining.
My read: with this little raw activity, there's barely any signal for the model to work
with either way, and it may be defaulting toward "not declining" when a page looks mostly
inactive rather than actively getting worse. These are hard cases because thin data cuts
both directions — a page could be declining into obscurity, or it could be new/niche and
was never very visible to begin with, and 1-2 impressions can't distinguish the two.

Suspiciously perfect check: after removing impressions_last_30d/prev_30d and their sibling
columns (identified as the direct source of trend_pct, which the label is derived from —
confirmed via a 1.000 correlation check), precision@K dropped from a leaky 1.0 at every K
to the honest 0.71-0.86 (LogReg) / 0.45-0.64 (RF) reported above. That earlier perfect
score was the leakage signal itself, not a good result.

In [7]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report

# --- Permutation importance on the stronger model ---
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    "feature": numeric_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
print("Top 10 features by permutation importance:")
print(importance_df.head(10).to_string(index=False))

# --- Classification report (context beyond just precision@K) ---
rf_preds = (rf_scores >= 0.5).astype(int)
print("\n", classification_report(y_test, rf_preds, digits=3))

# --- 3 concrete wrong cases: confident but wrong ---
test_df["rf_score"] = rf_scores
test_df["true_label"] = y_test.values
test_df["error"] = np.abs(test_df["rf_score"] - test_df["true_label"])
wrong_confident = test_df.sort_values("error", ascending=False).head(3)
cols_to_show = ["content_id", "true_label", "rf_score", "avg_position", "ctr", "days_since_last_update", "impressions_90d"]
print("\n3 most confidently wrong predictions:")
print(wrong_confident[cols_to_show].to_string(index=False))

Top 10 features by permutation importance:
              feature  importance_mean  importance_std
days_with_impressions         0.018538        0.002610
         avg_position         0.010471        0.002394
       log_clicks_90d         0.004399        0.001410
     log_sessions_90d         0.003556        0.000902
                  ctr         0.002839        0.000930
 engaged_sessions_90d         0.002713        0.000752
           clicks_90d         0.002614        0.000669
          scroll_rate         0.002066        0.000846
      engagement_rate         0.001476        0.000626
   days_with_sessions         0.001279        0.001113

               precision    recall  f1-score   support

           0      0.580     0.394     0.470      3440
           1      0.564     0.733     0.637      3675

    accuracy                          0.569      7115
   macro avg      0.572     0.563     0.553      7115
weighted avg      0.572     0.569     0.556      7115


3 most confidently wro

In [5]:
check = feat[["trend_pct", "impressions_last_30d", "impressions_prev_30d"]].copy()
check["computed_pct"] = ((check["impressions_last_30d"] - check["impressions_prev_30d"])
                          / check["impressions_prev_30d"].replace(0, np.nan)) * 100
print(check.corr(numeric_only=True))
print(check.head(10))

                      trend_pct  impressions_last_30d  impressions_prev_30d  \
trend_pct              1.000000              0.095971             -0.010575   
impressions_last_30d   0.095971              1.000000              0.863723   
impressions_prev_30d  -0.010575              0.863723              1.000000   
computed_pct           1.000000              0.096737             -0.010280   

                      computed_pct  
trend_pct                 1.000000  
impressions_last_30d      0.096737  
impressions_prev_30d     -0.010280  
computed_pct              1.000000  
   trend_pct  impressions_last_30d  impressions_prev_30d  computed_pct
0      -41.4                   578                   987    -41.438703
1      -57.7                  2501                  5915    -57.717667
2      -60.9                  2382                  6089    -60.880276
3      -13.8                  3626                  4206    -13.789824
4      -34.7                  4211                  6452    -34.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.